# Compression Strain Analysis — Interactive Notebook

This notebook computes **2D strain fields** from a compression-test video or image stack
using digital image correlation (DIC). It tracks how a sample deforms frame-by-frame,
builds a triangle mesh over the region you care about, and computes strain across it.

It is designed to run **anywhere with a web browser** via
[Google Colab](https://colab.research.google.com/) — no software installation, no coding
experience required. Just run the cells from top to bottom.

**It contains the exact same algorithm as the desktop program** (`strain_analysis_pkg`),
so results match. The algorithm code is shown in full below (Part 1) so you can read and
understand every step; the parts you actually interact with are Parts 2–7.

### How to use this notebook
1. Click **Runtime → Run all**, *or* run each cell in order with the ▶ button.
2. When you reach **Part 2**, upload your video/stack.
3. Adjust the **Settings** and **ROI** cells (they have easy sliders/boxes), re-running
   their preview cells until things look right.
4. Run the tracking and view your results with the interactive slider.
5. Download everything as a zip.

---
## Part 1 — The algorithm (read to understand; you don't edit these)

The cells in this part define the analysis functions. They are the real, unmodified
package code. You can read them to understand exactly how the tool works, but you do
**not** need to change anything here — just run each cell once.

In [ ]:
# Install the one package Colab doesn't ship with, then import everything we need.
!pip install tifffile --quiet

import os
import csv
from datetime import datetime

import numpy as np
import cv2
import tifffile
from scipy.spatial import Delaunay
from scipy.ndimage import map_coordinates, spline_filter
from matplotlib.path import Path
import matplotlib.pyplot as plt
import matplotlib as mpl

print("Setup complete.")

### 1a. Loading & preprocessing

Reads a TIFF stack or a video file into a stack of grayscale frames, then adjusts
contrast (stretching a chosen intensity range to full black→white) and optionally blurs
slightly to reduce noise.

In [ ]:
"""Image stack loading and preprocessing."""

import os

import cv2
import numpy as np
import tifffile as tiff


def load_image_stack(filename, frame_skip=1, end_frame=None):
    """Load a grayscale image stack from a TIFF file or a video file.

    Parameters
    ----------
    filename : str
        Path to a .tif/.tiff stack or a video file (.mov/.mp4/.avi/.mkv).
    frame_skip : int
        Keep every Nth frame.
    end_frame : int or None
        If given, trim the stack to frames [0, end_frame] after skipping.

    Returns
    -------
    imgs : np.ndarray, shape (n_frames, height, width), dtype matches source
    """
    ext = os.path.splitext(filename)[1].lower()

    if ext in (".tiff", ".tif"):
        imgs = tiff.imread(filename)
        if imgs.ndim == 4:
            imgs = np.array([cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in imgs])
        if imgs.ndim != 3:
            raise ValueError("TIFF must be a grayscale image stack")
    else:
        cap = cv2.VideoCapture(filename)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video file: {filename}")
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
        cap.release()
        if len(frames) == 0:
            raise ValueError(f"No frames read from: {filename}")
        imgs = np.array(frames)

    if frame_skip > 1:
        imgs = imgs[::frame_skip]

    if end_frame is not None:
        imgs = imgs[: end_frame + 1]

    return imgs


def adjust_contrast(img, low_in, high_in):
    """Linearly rescale [low_in, high_in] to [0, 255] and clip, as uint8."""
    img = np.clip((img.astype(np.float32) - low_in) / (high_in - low_in), 0, 1)
    return (img * 255).astype(np.uint8)


def preprocess_stack(imgs, low_in, high_in, gaussian_blur=0):
    """Apply contrast adjustment and optional Gaussian blur to every frame."""
    imgs = np.array([adjust_contrast(f, low_in, high_in) for f in imgs])

    if gaussian_blur > 0:
        ksize = gaussian_blur if gaussian_blur % 2 == 1 else gaussian_blur + 1
        imgs = np.array([cv2.GaussianBlur(f, (ksize, ksize), 0) for f in imgs])

    return imgs

### 1b. Mesh

Places a regular grid of tracking points inside your region of interest, connects them
into a triangle mesh (Delaunay triangulation), and sets up the arrays that will store
each point's position in every frame.

In [ ]:
"""Point grid generation within a polygon ROI, plus Delaunay triangulation."""

import numpy as np
from scipy.spatial import Delaunay


def generate_grid_points(roi_path, spacing):
    """Generate a regular grid of (row, col) points inside a polygon ROI.

    Parameters
    ----------
    roi_path : matplotlib.path.Path
        Polygon ROI, as returned by roi.select_polygon_roi.
    spacing : int
        Grid spacing in pixels.

    Returns
    -------
    pts : list of (row, col) tuples
        Grid points that fall inside the ROI.
    """
    vertices = np.array(roi_path.vertices)
    xmin, ymin = np.min(vertices, axis=0)
    xmax, ymax = np.max(vertices, axis=0)

    pts = []
    for r in np.arange(int(ymin), int(ymax), spacing):
        for c in np.arange(int(xmin), int(xmax), spacing):
            if roi_path.contains_point((c, r)):
                pts.append((int(r), int(c)))

    return pts


def triangulate_points(pts):
    """Delaunay-triangulate a list of (row, col) points.

    Parameters
    ----------
    pts : list of (row, col) tuples

    Returns
    -------
    tri : scipy.spatial.Delaunay
        Triangulation; tri.simplices gives point-index triples.
    """
    xy = np.array([[p[1], p[0]] for p in pts])
    return Delaunay(xy)


def init_tracks(pts, n_frames, start_frame):
    """Allocate NaN-filled displacement arrays and seed the start frame.

    Parameters
    ----------
    pts : list of (row, col) tuples
    n_frames : int
    start_frame : int

    Returns
    -------
    XA, YA : np.ndarray, shape (len(pts), n_frames)
        Row (XA) and column (YA) position of each point at each frame,
        NaN where not yet tracked.
    """
    n_pts = len(pts)
    XA = np.full((n_pts, n_frames), np.nan)
    YA = np.full((n_pts, n_frames), np.nan)

    for i, (r, c) in enumerate(pts):
        XA[i, start_frame] = r
        YA[i, start_frame] = c

    return XA, YA

### 1c. Strain

Computes strain at each point using a **local least-squares fit** over its mesh
neighbors (more robust to noise than using a single triangle), then averages to a
per-triangle value for the colored overlay. NaN-aware: points that couldn't be tracked
are excluded rather than corrupting their neighbors.

In [ ]:
"""Strain computation: local least-squares displacement gradient, NaN-aware.

The original script computed strain per-triangle directly from a Delaunay simplex's 3
vertices (a "constant strain triangle" / CST calculation). That's exact for a perfectly
rigid triangle, but any single vertex's tracking noise goes straight into that triangle's
strain value with no averaging -- a noisy point corrupts every triangle touching it.

Here, strain is instead estimated at each *point* from a local least-squares fit over all
of its mesh neighbors (typically 4-8 points, not just 2). This is the same small-strain
displacement-gradient math, just over-determined instead of exactly-determined, so a
single noisy neighbor is damped rather than directly propagated. Per-triangle strain
(still needed for the mesh-overlay visualization) is then just the average of its 3
vertices' point strains.

Convention (kept identical to the original script for compatibility): "x" refers to the
column direction (YA), "y" refers to the row direction (XA). ex = du/dx, ey = dv/dy,
gxy = du/dy + dv/dx.
"""

import numpy as np


def build_neighbor_lists(n_pts, tri):
    """One-ring neighbor point-indices for every point, from a Delaunay triangulation's
    edges.
    """
    neighbors = [set() for _ in range(n_pts)]
    for simplex in tri.simplices:
        for a, b in ((0, 1), (1, 2), (2, 0)):
            i, j = simplex[a], simplex[b]
            neighbors[i].add(j)
            neighbors[j].add(i)
    return [np.array(sorted(s), dtype=int) for s in neighbors]


def _fit_point_strain(dx, dy, du, dv):
    """Least-squares fit of [ex, exy_from_u] from du = ex*dx + exy_from_u*dy, and
    [exy_from_v, ey] from dv = exy_from_v*dx + ey*dy. Returns (ex, ey, gxy) or None if
    the system is degenerate (e.g. all neighbors collinear).
    """
    A = np.column_stack([dx, dy])
    try:
        coef_u, *_ = np.linalg.lstsq(A, du, rcond=None)
        coef_v, *_ = np.linalg.lstsq(A, dv, rcond=None)
    except np.linalg.LinAlgError:
        return None

    ex = coef_u[0]
    ey = coef_v[1]
    gxy = coef_u[1] + coef_v[0]
    return ex, ey, gxy


def compute_point_strain(pts, tri, XA, YA, start_frame, min_neighbors=3):
    """Local least-squares strain at every point, for every frame.

    Parameters
    ----------
    pts : list of (row, col) tuples
    tri : scipy.spatial.Delaunay
    XA, YA : np.ndarray, shape (n_pts, n_frames)
        Row/col position of each point per frame (NaN where untracked), as produced by
        tracking.track_sequence.
    start_frame : int
        Reference frame; strain is relative to positions at this frame.
    min_neighbors : int
        Minimum number of valid (non-NaN) neighbors required to fit strain at a point.
        Below this, the point's strain is NaN for that frame.

    Returns
    -------
    point_strain : np.ndarray, shape (n_pts, n_frames, 3)
        (ex, ey, gxy) per point per frame; NaN where not computable.
    """
    n_pts, n_frames = XA.shape
    neighbor_lists = build_neighbor_lists(n_pts, tri)

    x_ref = YA[:, start_frame]  # column = "x"
    y_ref = XA[:, start_frame]  # row = "y"

    point_strain = np.full((n_pts, n_frames, 3), np.nan)

    for k in range(start_frame, n_frames):
        x_cur = YA[:, k]
        y_cur = XA[:, k]

        for i in range(n_pts):
            if np.isnan(x_ref[i]) or np.isnan(x_cur[i]):
                continue

            nbrs = neighbor_lists[i]
            if nbrs.size == 0:
                continue

            valid = ~np.isnan(x_ref[nbrs]) & ~np.isnan(x_cur[nbrs])
            nbrs = nbrs[valid]
            if nbrs.size < min_neighbors:
                continue

            dx = x_ref[nbrs] - x_ref[i]
            dy = y_ref[nbrs] - y_ref[i]
            du = (x_cur[nbrs] - x_cur[i]) - dx
            dv = (y_cur[nbrs] - y_cur[i]) - dy

            fit = _fit_point_strain(dx, dy, du, dv)
            if fit is None:
                continue

            point_strain[i, k, :] = fit

    return point_strain


def compute_triangle_strain(tri, point_strain, min_valid_vertices=2):
    """Per-triangle strain (for mesh-overlay visualization), averaged from vertex
    point-strains. A triangle needs at least `min_valid_vertices` non-NaN vertices to
    produce a value; otherwise it's NaN for that frame.

    Returns
    -------
    tri_strain : np.ndarray, shape (n_frames, n_triangles, 3)
    """
    n_pts, n_frames, _ = point_strain.shape
    simplices = tri.simplices
    n_tri = len(simplices)

    tri_strain = np.full((n_frames, n_tri, 3), np.nan)

    for k in range(n_frames):
        for t, simplex in enumerate(simplices):
            vals = point_strain[simplex, k, :]  # (3, 3)
            valid_mask = ~np.isnan(vals[:, 0])
            if valid_mask.sum() < min_valid_vertices:
                continue
            tri_strain[k, t, :] = np.nanmean(vals, axis=0)

    return tri_strain

### 1d. Tracking

The core of the tool. For each point it finds where that patch of the sample moved to in
the next frame:

- **Reference-frame tracking** — each point is matched back to a template from an anchor
  frame (not the previous frame), which avoids drift building up over a long sequence.
- **KLT subpixel refinement** — refines the match to a small fraction of a pixel.
- **Honest gaps** — if a point genuinely can't be matched, it's marked as missing
  (not faked), and (optionally) *estimated* from neighbors that did track — flagged with
  confidence `-1` so estimates stay distinguishable from real measurements.

In [ ]:
"""Point tracking: reference-frame anchored coarse match + KLT subpixel refinement.

Design (vs. the original single-file script's frame-to-frame block matching):

1. Reference-frame tracking. Each point is matched against a template taken from an
   "anchor" frame (initially the start frame), not the previous frame. This avoids the
   drift that comes from compounding frame-to-frame errors. The anchor is only moved
   forward ("re-anchored") when match confidence drops, so the template stays valid
   without accumulating error every single frame.

2. KLT translation subpixel refinement. A coarse integer/parabolic match
   (cv2.matchTemplate, same idea as the original script) gives a starting guess, which
   is then refined with a Lucas-Kanade (KLT) translation-only Newton iteration. This
   converges to a small fraction of a pixel, well beyond the accuracy of a 3-point
   parabola fit alone. (A full affine per-point refinement was evaluated first but is
   ill-conditioned at typical patch sizes and diverges; local strain is instead computed
   from the tracked point mesh in strain.py.)

3. Explicit confidence, no silent failures. A failed or low-confidence match produces
   NaN, not a fake (0, 0) displacement -- so it can be excluded from strain computation
   instead of silently corrupting neighboring triangles.

4. Optional gap-filling via neighbor extrapolation. If a point can't find a real match,
   its position can instead be *estimated* from a local affine fit over its mesh
   neighbors that DID track successfully that frame (see `_extrapolate_position`). This
   is flagged with confidence -1 (never a value a real match can produce) so estimated
   points remain distinguishable from measured ones downstream -- full ROI coverage
   without silently pretending an estimate is a measurement.
"""

import cv2
import numpy as np
from scipy.ndimage import map_coordinates, spline_filter


ESTIMATED_CONFIDENCE = -1.0  # sentinel: position is a neighbor-extrapolated estimate,
                             # not a real match. Real confidences are always >= corr_threshold > 0.


def _extract_patch(img, r0, c0, half_h, half_w):
    """Extract an axis-aligned, odd-sized patch centered exactly on the pixel nearest
    (r0, c0). Size is (2*half_h+1, 2*half_w+1) so the center pixel is unambiguous --
    an even-sized patch has no single center pixel, which introduces a systematic
    0.5px bias into any refinement anchored on it. Returns None if out of bounds.
    """
    r0i, c0i = int(round(r0)), int(round(c0))
    r_start, r_end = r0i - half_h, r0i + half_h + 1
    c_start, c_end = c0i - half_w, c0i + half_w + 1

    if r_start < 0 or c_start < 0 or r_end > img.shape[0] or c_end > img.shape[1]:
        return None

    return img[r_start:r_end, c_start:c_end]


def _extrapolate_position(ref_row_i, ref_col_i, ref_rows_nbrs, ref_cols_nbrs, cur_rows_nbrs, cur_cols_nbrs):
    """Estimate point i's current position from a local affine fit over neighbors that
    DID track successfully this frame.

    Fits current = A @ [ref_row, ref_col, 1] (least squares) using the neighbors' own
    reference -> current mappings, then evaluates that fit at point i's reference
    position. Requires >=3 non-collinear neighbors; returns None if the fit is
    degenerate (e.g. neighbors happen to be collinear).
    """
    ones = np.ones_like(ref_rows_nbrs, dtype=np.float64)
    design = np.column_stack([ref_rows_nbrs, ref_cols_nbrs, ones])
    query = np.array([ref_row_i, ref_col_i, 1.0])

    try:
        row_coef, *_ = np.linalg.lstsq(design, cur_rows_nbrs, rcond=None)
        col_coef, *_ = np.linalg.lstsq(design, cur_cols_nbrs, rcond=None)
    except np.linalg.LinAlgError:
        return None

    est_row = float(query @ row_coef)
    est_col = float(query @ col_coef)

    if not (np.isfinite(est_row) and np.isfinite(est_col)):
        return None

    return est_row, est_col


def _prefilter_frame(img):
    """Precompute cubic-spline coefficients for a whole frame, once, for reuse across
    every point's KLT refinement on that frame (see `_translation_refine`).
    """
    return spline_filter(img.astype(np.float64), order=3)


def _coarse_match(template, search_patch, t_x, t_y, corr_threshold):
    """Integer + parabolic-subpixel template match, in the spirit of the original script.

    Returns (dr, dc, confidence) offset of the template center within search_patch,
    or None if the match is unreliable.
    """
    h, w = template.shape
    if search_patch.shape[0] < h or search_patch.shape[1] < w:
        return None
    if np.all(template == 0):
        return None

    result = cv2.matchTemplate(search_patch, template, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, max_loc = cv2.minMaxLoc(result)

    if max_val < corr_threshold:
        return None

    mc, mr = max_loc  # cv2 returns (x=col, y=row)
    rh, rw = result.shape
    sub_r, sub_c = float(mr), float(mc)

    if 0 < mr < rh - 1:
        a, b, c = result[mr - 1, mc], result[mr, mc], result[mr + 1, mc]
        denom = 2 * b - a - c
        if abs(denom) > 1e-12:
            sub_r = mr + (a - c) / (2 * denom)

    if 0 < mc < rw - 1:
        a, b, c = result[mr, mc - 1], result[mr, mc], result[mr, mc + 1]
        denom = 2 * b - a - c
        if abs(denom) > 1e-12:
            sub_c = mc + (a - c) / (2 * denom)

    dr = -t_x + sub_r
    dc = -t_y + sub_c
    return dr, dc, float(max_val)


def _translation_refine(template, cur_img_coeffs, guess_row, guess_col, max_iters=20, eps=1e-3):
    """Lucas-Kanade (KLT) translation-only subpixel refinement.

    Fits a subpixel translation (d_row, d_col) so that sampling cur_img at
    (guess_row + d_row, guess_col + d_col) best matches `template`, using the classic
    KLT normal-equations update (Newton step on local image gradients).

    A full affine (6-parameter) formulation was tried first but is ill-conditioned for
    small patches -- the translation/shear/scale terms are too correlated at this patch
    size and it diverges instead of converging. Translation-only is well-conditioned and
    standard practice for DIC subpixel refinement; per-triangle strain is computed
    separately (see strain.py) from the resulting point positions, so nothing is lost.

    `cur_img_coeffs` must already be the cubic-spline-prefiltered coefficients of the
    current frame (see `_prefilter_frame`) -- computing that filter is O(image size), so
    it must be done once per frame by the caller, not once per point/iteration here
    (map_coordinates' default `prefilter=True` would otherwise silently redo it on the
    full image on every single call, which for a few hundred points times ~20 iterations
    each turns into the dominant cost of the whole tracking pass).

    Returns ((d_row, d_col), zncc) or None on failure/non-convergence, where zncc is the
    final zero-normalized cross-correlation (comparable in scale to
    cv2.TM_CCOEFF_NORMED's output).
    """
    h, w = template.shape
    rc = (h - 1) / 2.0
    cc_ = (w - 1) / 2.0
    rr, cc = np.meshgrid(np.arange(h) - rc, np.arange(w) - cc_, indexing="ij")

    d_row, d_col = 0.0, 0.0
    sampled = None

    for _ in range(max_iters):
        Wr = guess_row + d_row + rr
        Wc = guess_col + d_col + cc
        sampled = map_coordinates(
            cur_img_coeffs, [Wr.ravel(), Wc.ravel()], order=3, mode="constant", cval=np.nan,
            prefilter=False,
        ).reshape(h, w)
        if np.isnan(sampled).any():
            return None

        gr, gc = np.gradient(sampled)
        A = np.array(
            [
                [np.sum(gr * gr), np.sum(gr * gc)],
                [np.sum(gr * gc), np.sum(gc * gc)],
            ]
        )
        b = np.array([np.sum(gr * (template - sampled)), np.sum(gc * (template - sampled))])

        try:
            delta = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        d_row += float(delta[0])
        d_col += float(delta[1])

        if np.linalg.norm(delta) < eps:
            break

    if sampled is None:
        return None

    t_mean, t_std = template.mean(), template.std() + 1e-8
    s_mean, s_std = sampled.mean(), sampled.std() + 1e-8
    zncc = float(np.mean((sampled - s_mean) * (template - t_mean)) / (s_std * t_std))

    return (d_row, d_col), zncc


def track_point(
    ref_img,
    ref_row,
    ref_col,
    cur_img,
    cur_img_coeffs,
    guess_row,
    guess_col,
    half_h,
    half_w,
    t_x,
    t_y,
    corr_threshold,
    max_disp,
    lk_max_iters=20,
    lk_eps=1e-3,
):
    """Track one point from a reference-frame template into cur_img.

    Combines a coarse cv2.matchTemplate search (bounded by +-t_x/+-t_y) with a KLT
    translation subpixel refinement. `cur_img_coeffs` is cur_img's precomputed
    spline-filter coefficients (see `_prefilter_frame`) -- pass the same one in for every
    point tracked against this frame. Returns a dict with the new position and match
    confidence, or None if the point could not be reliably tracked.
    """
    template = _extract_patch(ref_img, ref_row, ref_col, half_h, half_w)
    if template is None or template.size == 0 or np.all(template == 0):
        return None

    search_patch = _extract_patch(cur_img, guess_row, guess_col, half_h + t_x, half_w + t_y)
    if search_patch is None:
        return None

    coarse = _coarse_match(template, search_patch, t_x, t_y, corr_threshold)
    if coarse is None:
        return None
    dr_coarse, dc_coarse, coarse_conf = coarse

    coarse_row = guess_row + dr_coarse
    coarse_col = guess_col + dc_coarse

    refined = _translation_refine(
        template.astype(np.float64),
        cur_img_coeffs,
        coarse_row,
        coarse_col,
        max_iters=lk_max_iters,
        eps=lk_eps,
    )

    if refined is None or refined[1] < corr_threshold:
        new_row, new_col, confidence = coarse_row, coarse_col, coarse_conf
    else:
        (d_row, d_col), zncc = refined
        new_row, new_col, confidence = coarse_row + d_row, coarse_col + d_col, zncc

    step_disp = float(np.hypot(new_row - guess_row, new_col - guess_col))
    if step_disp > max_disp:
        return None

    return {"row": new_row, "col": new_col, "confidence": confidence}


def track_sequence(
    imgs,
    pts,
    XA,
    YA,
    start_frame,
    delta_x,
    delta_y,
    t_x,
    t_y,
    corr_threshold,
    max_disp,
    reanchor_threshold=None,
    lk_max_iters=20,
    tri=None,
    fill_gaps=True,
    min_neighbors_fill=3,
    verbose=True,
):
    """Track all points across the full image sequence, in place on XA/YA.

    Points are tracked against a reference-frame template that only moves forward
    ("re-anchors") once its match confidence drops below `reanchor_threshold`.

    If `tri` (the mesh's Delaunay triangulation) is given and `fill_gaps=True`, a point
    that fails a real match is NOT immediately left as a permanent gap: its position is
    instead *estimated* via `_extrapolate_position` from whichever of its mesh neighbors
    DID get a real match this frame (never from other estimated neighbors, so estimates
    never chain off of other estimates). This gives full ROI coverage, but an estimated
    point is flagged with confidence == ESTIMATED_CONFIDENCE (-1) rather than looking
    like a real measurement, and it keeps using its last real anchor template next
    frame -- so if real tracking becomes possible again later, it can recover on its
    own instead of being reseeded from a guess. A point only ends up NaN if it fails a
    real match AND doesn't have >= min_neighbors_fill real neighbors to extrapolate
    from (or `tri` wasn't provided / `fill_gaps=False`).

    Returns
    -------
    XA, YA : the same arrays passed in, updated in place
    CA : np.ndarray, shape (n_pts, n_frames)
        Per-point, per-frame confidence: real match confidence (>= corr_threshold),
        ESTIMATED_CONFIDENCE (-1) if extrapolated, or 0 if neither was possible.
    """
    n_pts, n_frames = XA.shape
    half_h, half_w = delta_x // 2, delta_y // 2

    if reanchor_threshold is None:
        reanchor_threshold = min(0.99, corr_threshold + 0.1)

    do_fill = fill_gaps and tri is not None
    neighbor_lists = build_neighbor_lists(n_pts, tri) if do_fill else None

    CA = np.zeros((n_pts, n_frames))
    CA[:, start_frame] = 1.0

    anchor_frame = np.full(n_pts, start_frame, dtype=int)

    total = n_frames - 1 - start_frame
    for k in range(start_frame, n_frames - 1):
        if verbose:
            print(f"\rTracking frame {k - start_frame + 1}/{total}", end="", flush=True)

        cur_img = imgs[k + 1]
        cur_img_coeffs = _prefilter_frame(cur_img)  # once per frame, not once per point

        failed = []

        for i in range(n_pts):
            guess_row, guess_col = XA[i, k], YA[i, k]
            if np.isnan(guess_row) or np.isnan(guess_col):
                failed.append(i)
                continue

            af = anchor_frame[i]
            ref_row, ref_col = XA[i, af], YA[i, af]
            ref_img = imgs[af]

            result = track_point(
                ref_img,
                ref_row,
                ref_col,
                cur_img,
                cur_img_coeffs,
                guess_row,
                guess_col,
                half_h,
                half_w,
                t_x,
                t_y,
                corr_threshold,
                max_disp,
                lk_max_iters=lk_max_iters,
            )

            if result is None:
                failed.append(i)
                continue

            XA[i, k + 1] = result["row"]
            YA[i, k + 1] = result["col"]
            CA[i, k + 1] = result["confidence"]

            if result["confidence"] < reanchor_threshold:
                anchor_frame[i] = k + 1

        if do_fill:
            for i in failed:
                nbrs = neighbor_lists[i]
                if nbrs.size == 0:
                    XA[i, k + 1], YA[i, k + 1], CA[i, k + 1] = np.nan, np.nan, 0.0
                    continue

                real = nbrs[CA[nbrs, k + 1] > 0]  # only real matches, never estimates
                if real.size < min_neighbors_fill:
                    XA[i, k + 1], YA[i, k + 1], CA[i, k + 1] = np.nan, np.nan, 0.0
                    continue

                est = _extrapolate_position(
                    XA[i, start_frame], YA[i, start_frame],
                    XA[real, start_frame], YA[real, start_frame],
                    XA[real, k + 1], YA[real, k + 1],
                )
                if est is None:
                    XA[i, k + 1], YA[i, k + 1], CA[i, k + 1] = np.nan, np.nan, 0.0
                    continue

                XA[i, k + 1], YA[i, k + 1] = est
                CA[i, k + 1] = ESTIMATED_CONFIDENCE
                # anchor_frame[i] is deliberately left untouched: an estimate isn't a
                # real texture confirmation, so the next frame still tries to match
                # against the last genuinely-confirmed template.
        else:
            for i in failed:
                XA[i, k + 1], YA[i, k + 1], CA[i, k + 1] = np.nan, np.nan, 0.0

    if verbose:
        print()

    return XA, YA, CA

### 1e. Exporting & drawing

Builds the colored strain overlay on each frame, writes the results CSV, the
strain-over-time plot, and the overlay video.

In [ ]:
"""CSV / plot / video-overlay export of tracking and strain results."""

import csv
import os
from datetime import datetime

import cv2
import numpy as np


def select_strain_component(data, strain_mode):
    """Pick out ex, ey, gxy, or von Mises from a (..., 3) strain array.

    Works for both point_strain (n_pts, n_frames, 3) and tri_strain
    (n_frames, n_triangles, 3) since only the last axis is interpreted.
    """
    ex, ey, gxy = data[..., 0], data[..., 1], data[..., 2]

    if strain_mode == "ex":
        return ex, "εx (Horizontal Strain)"
    elif strain_mode == "ey":
        return ey, "εy (Vertical Strain)"
    elif strain_mode == "gxy":
        return gxy, "γxy (Shear Strain)"
    else:
        vonmises = np.sqrt(ex**2 - ex * ey + ey**2 + 0.75 * gxy**2)
        return vonmises, "Von Mises Strain"


def make_export_dir(script_dir, filename):
    """Create a timestamped results/<basename>_<timestamp>/ directory."""
    base_name = os.path.splitext(os.path.basename(filename))[0]
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_name = f"{base_name}_{timestamp}"
    export_dir = os.path.join(script_dir, "results", run_name)
    os.makedirs(export_dir, exist_ok=True)
    return export_dir, base_name


def export_strain_csv(export_dir, base_name, point_strain, confidence, XA, YA, start_frame):
    """Write per-point, per-frame strain + tracking confidence to CSV.

    Columns: frame, point, row, col, ex, ey, gxy, confidence

    confidence is the real match correlation (>= corr_threshold) for a genuinely
    tracked point, or tracking.ESTIMATED_CONFIDENCE (-1) if the position was instead
    estimated from neighbors because no real match was found that frame (see
    tracking.track_sequence's fill_gaps option) -- filter on confidence < 0 to exclude
    estimated points from downstream analysis if you only want real measurements.
    """
    n_pts, n_frames, _ = point_strain.shape
    csv_path = os.path.join(export_dir, f"{base_name}_strain.csv")

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["frame", "point", "row", "col", "ex", "ey", "gxy", "confidence"])
        for k in range(start_frame, n_frames):
            for i in range(n_pts):
                ex_val, ey_val, gxy_val = point_strain[i, k, :]
                if np.isnan(ex_val):
                    continue
                writer.writerow(
                    [
                        k,
                        i,
                        f"{XA[i, k]:.3f}",
                        f"{YA[i, k]:.3f}",
                        f"{ex_val:.6f}",
                        f"{ey_val:.6f}",
                        f"{gxy_val:.6f}",
                        f"{confidence[i, k]:.4f}",
                    ]
                )

    print(f"Strain CSV saved: {csv_path}")
    return csv_path


def plot_strain_over_time(export_dir, base_name, point_strain, strain_mode, start_frame):
    """Save a PNG of mean strain (over all valid points) vs. frame number."""
    import matplotlib.pyplot as plt

    plot_data, plot_label = select_strain_component(point_strain, strain_mode)
    n_frames = plot_data.shape[1]
    mean_strain = np.nanmean(plot_data, axis=0)
    frame_nums = np.arange(n_frames)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(frame_nums[start_frame:], mean_strain[start_frame:], linewidth=1.5)
    ax.set_xlabel("Frame")
    ax.set_ylabel(f"Mean {plot_label}")
    ax.set_title("Strain Over Time")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    plot_path = os.path.join(export_dir, f"{base_name}_strain_over_time.png")
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)

    print(f"Strain plot saved: {plot_path}")
    return plot_path


def render_overlay_frame(img, tri, tri_strain_frame, pts_now, vmin, vmax, cmap, show_triangle_edges):
    """Render one strain-colored triangle-mesh overlay frame on top of a grayscale image.

    Returns a BGR uint8 image ready for cv2.VideoWriter or display.
    """
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    overlay = img_rgb.copy()

    for t, simplex in enumerate(tri.simplices):
        val = tri_strain_frame[t]
        if np.isnan(val):
            continue
        # A triangle can have a valid *strain* value averaged from only 2 of its 3
        # vertices (see strain.compute_triangle_strain's min_valid_vertices), but if any
        # vertex's tracked *position* is NaN, drawing the polygon would cast NaN -> a
        # garbage int coordinate and produce a line shooting across the frame. Skip
        # rendering (not just strain) unless every vertex position is valid this frame.
        if np.isnan(pts_now[simplex]).any():
            continue
        norm = (val - vmin) / (vmax - vmin + 1e-12)
        color = cmap(norm)[:3]
        color = tuple(int(255 * c) for c in color)
        poly_pts = pts_now[simplex].astype(int)
        cv2.fillPoly(overlay, [poly_pts], color)
        if show_triangle_edges:
            cv2.polylines(overlay, [poly_pts], True, (0, 0, 0), 1)

    blended = cv2.addWeighted(overlay, 0.45, img_rgb, 0.55, 0)
    return cv2.cvtColor(blended, cv2.COLOR_RGB2BGR)


def export_strain_video(
    export_dir,
    base_name,
    imgs,
    XA,
    YA,
    tri,
    tri_strain,
    strain_mode,
    show_triangle_edges,
    start_frame,
    fps=15,
):
    """Save an MP4 with the strain-colored mesh overlaid on every frame."""
    import matplotlib.pyplot as plt

    export_data, _ = select_strain_component(tri_strain, strain_mode)
    vmin = np.nanpercentile(export_data, 5)
    vmax = np.nanpercentile(export_data, 95)
    cmap = plt.cm.jet

    n_frames = imgs.shape[0]
    mm, nn = imgs.shape[1], imgs.shape[2]
    n_pts = XA.shape[0]

    video_path = os.path.join(export_dir, f"{base_name}_strain.mp4")
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_out = cv2.VideoWriter(video_path, fourcc, fps, (nn, mm))

    total = n_frames - start_frame
    for k in range(start_frame, n_frames):
        print(f"\rExporting frame {k - start_frame + 1}/{total}", end="", flush=True)
        pts_now = np.array([[YA[i, k], XA[i, k]] for i in range(n_pts)])
        frame_out = render_overlay_frame(
            imgs[k], tri, export_data[k], pts_now, vmin, vmax, cmap, show_triangle_edges
        )
        video_out.write(frame_out)

    video_out.release()
    print(f"\nStrain video saved: {video_path}")
    return video_path

---
## Part 2 — Upload your video or image stack

Run the cell below and pick your file (`.tif`, `.tiff`, `.mov`, `.mp4`, `.avi`, `.mkv`).
Large files may take a moment to upload.

In [ ]:
from google.colab import files

uploaded = files.upload()
VIDEO_FILENAME = list(uploaded.keys())[0]
print("Uploaded:", VIDEO_FILENAME)

---
## Part 3 — Settings

Adjust these with the boxes/menus on the right (Colab shows them as a form). You can
re-run this cell any time after changing values. Hover notes explain each one; the
defaults are a good starting point.

- **start_frame / end_frame** — which frames to analyze (`end_frame = -1` means "to the end").
- **strain_mode** — which strain to display (`vonmises` is a good general choice).
- **mesh_spacing** — spacing between tracking points in pixels (smaller = finer, slower).
- **corr_threshold** — how confident a match must be to be accepted (0–1; lower = more
  coverage but more risk of bad matches).
- **max_disp** — largest allowed per-frame movement in pixels.
- **fill_gaps** — estimate untrackable points from their neighbors for full coverage.

In [ ]:
#@title Analysis Settings { display-mode: "form" }
#@markdown **Frames**
start_frame = 0  #@param {type:"integer"}
end_frame = -1   #@param {type:"integer"}
frame_skip = 1   #@param {type:"integer"}
#@markdown **Strain**
strain_mode = "vonmises"  #@param ["ex", "ey", "gxy", "vonmises"]
min_neighbors = 3  #@param {type:"integer"}
#@markdown **Mesh**
mesh_spacing = 20  #@param {type:"integer"}
show_triangle_edges = True  #@param {type:"boolean"}
#@markdown **Tracking**
delta_x = 30  #@param {type:"integer"}
delta_y = 30  #@param {type:"integer"}
t_x = 25  #@param {type:"integer"}
t_y = 25  #@param {type:"integer"}
corr_threshold = 0.85  #@param {type:"number"}
max_disp = 30  #@param {type:"number"}
gaussian_blur = 3  #@param {type:"integer"}
fill_gaps = True  #@param {type:"boolean"}
#@markdown **Contrast**
low_in = 28  #@param {type:"integer"}
high_in = 250  #@param {type:"integer"}

print("Settings loaded.")

### Load the frames and preview the starting frame

Run this after setting your options above. This is the frame you'll draw your ROI on in
Part 4 — check that your sample is clearly visible and the contrast looks good.

In [ ]:
_end = None if end_frame < 0 else end_frame
imgs = load_image_stack(VIDEO_FILENAME, frame_skip=frame_skip, end_frame=_end)
imgs = preprocess_stack(imgs, low_in, high_in, gaussian_blur)
n_frames, height, width = imgs.shape
print(f"Loaded {n_frames} frames, each {width} x {height} pixels.")

plt.figure(figsize=(9, 9))
plt.imshow(imgs[start_frame], cmap="gray")
plt.title(f"Start frame ({start_frame}) — read off coordinates here for your ROI")
plt.show()

---
## Part 4 — Draw the region of interest (ROI) by clicking

Run the two cells below. An image of your start frame will appear that you can **click on
directly** to draw a polygon around the part of the sample you want to analyze:

- **Click** on the image to drop each corner point.
- **Undo** removes the last point; **Clear** starts over.
- **Finish ROI** when you've placed at least 3 points (go around the sample, then Finish).

Draw *inside* your sample, and — importantly — **keep the outline clear of the edges where
the sample touches the grips/tweezers**, since those regions don't track reliably in
single-camera 2D.

*(The first cell just sets up the drawing tool — run it once. The second cell is the one
you actually click on; you can re-run it any time to redraw.)*

In [ ]:
# --- Setup for the click-to-draw ROI tool (run once) -----------------------------------
# Uses a small HTML canvas so you can click points directly on the image in Colab.
import base64
from IPython.display import Javascript, display

_ROI_CANVAS_JS = r'''
async function drawPolygonRoi(imgB64, dispW, dispH, scale) {
  const box = document.createElement('div');
  document.body.appendChild(box);

  const info = document.createElement('div');
  info.innerHTML = '<b>Click to place points around your sample</b> (need at least 3), then click <b>Finish ROI</b>.';
  info.style.marginBottom = '6px';
  box.appendChild(info);

  const canvas = document.createElement('canvas');
  canvas.width = dispW; canvas.height = dispH;
  canvas.style.border = '1px solid #888';
  canvas.style.cursor = 'crosshair';
  box.appendChild(canvas);
  const ctx = canvas.getContext('2d');

  const img = new Image();
  img.src = 'data:image/png;base64,' + imgB64;
  await new Promise((r) => { img.onload = r; });

  let pts = [];
  function redraw() {
    ctx.clearRect(0, 0, dispW, dispH);
    ctx.drawImage(img, 0, 0, dispW, dispH);
    if (pts.length > 0) {
      ctx.beginPath();
      ctx.moveTo(pts[0][0], pts[0][1]);
      for (let i = 1; i < pts.length; i++) ctx.lineTo(pts[i][0], pts[i][1]);
      ctx.closePath();
      ctx.strokeStyle = '#39FF14'; ctx.lineWidth = 2; ctx.stroke();
      for (const p of pts) {
        ctx.beginPath(); ctx.arc(p[0], p[1], 4, 0, 2 * Math.PI);
        ctx.fillStyle = '#39FF14'; ctx.fill();
      }
    }
  }
  redraw();

  canvas.onclick = (e) => {
    const rect = canvas.getBoundingClientRect();
    pts.push([e.clientX - rect.left, e.clientY - rect.top]);
    redraw();
  };

  const bar = document.createElement('div');
  bar.style.marginTop = '6px';
  box.appendChild(bar);
  function mkBtn(label) {
    const b = document.createElement('button');
    b.textContent = label;
    b.style.marginRight = '6px'; b.style.padding = '4px 10px';
    bar.appendChild(b); return b;
  }
  const undoBtn = mkBtn('Undo');
  const clearBtn = mkBtn('Clear');
  const finishBtn = mkBtn('Finish ROI');
  undoBtn.onclick = () => { pts.pop(); redraw(); };
  clearBtn.onclick = () => { pts = []; redraw(); };

  await new Promise((resolve) => {
    finishBtn.onclick = () => {
      if (pts.length < 3) { alert('Please place at least 3 points first.'); return; }
      resolve();
    };
  });

  box.remove();
  // Map click coordinates (display pixels) back to original image pixels.
  return pts.map((p) => [p[0] / scale, p[1] / scale]);
}
'''


def click_polygon_roi(frame, max_width=800):
    from google.colab.output import eval_js  # Colab-only; imported here so the fallback still loads elsewhere
    h, w = frame.shape[:2]
    scale = min(1.0, max_width / float(w))
    disp_w, disp_h = int(round(w * scale)), int(round(h * scale))
    ok, buf = cv2.imencode('.png', frame)
    b64 = base64.b64encode(buf).decode()
    display(Javascript(_ROI_CANVAS_JS))
    coords = eval_js('drawPolygonRoi("%s", %d, %d, %f)' % (b64, disp_w, disp_h, scale))
    verts = [(float(x), float(y)) for x, y in coords]
    return Path(verts), verts


# Fallback used only by the optional cell at the end of Part 4.
def make_ellipse_roi(cx, cy, rx, ry, n=48):
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    verts = [(cx + rx * np.cos(a), cy + ry * np.sin(a)) for a in t]
    return Path(verts), verts

print("ROI drawing tool ready.")

### Click to draw your ROI, then confirm
Run this cell, click points on the image, and press **Finish ROI**. A preview of the result appears below. Re-run to redraw.

In [ ]:
roi_path, roi_vertices = click_polygon_roi(imgs[start_frame])

vx = [v[0] for v in roi_vertices] + [roi_vertices[0][0]]
vy = [v[1] for v in roi_vertices] + [roi_vertices[0][1]]
plt.figure(figsize=(9, 9))
plt.imshow(imgs[start_frame], cmap="gray")
plt.plot(vx, vy, "-", color="#39FF14", linewidth=2)
plt.title(f"Your ROI ({len(roi_vertices)} points) — re-run the cell to redraw")
plt.show()

<details>
<summary><b>Fallback:</b> set the ROI by numbers instead of clicking (only if the click tool doesn't work)</summary>

If the click canvas above doesn't appear or respond in your environment, uncomment and run
the cell below to define an **ellipse** ROI from a center and radii instead. Adjust the
numbers and re-run until the green outline fits.
</details>

In [ ]:
# OPTIONAL fallback — leave commented unless the click tool above didn't work.
# center_x, center_y, radius_x, radius_y = 715, 355, 110, 145
# roi_path, roi_vertices = make_ellipse_roi(center_x, center_y, radius_x, radius_y)
#
# vx = [v[0] for v in roi_vertices] + [roi_vertices[0][0]]
# vy = [v[1] for v in roi_vertices] + [roi_vertices[0][1]]
# plt.figure(figsize=(9, 9))
# plt.imshow(imgs[start_frame], cmap="gray")
# plt.plot(vx, vy, "-", color="#39FF14", linewidth=2)
# plt.title("Ellipse ROI preview — adjust numbers and re-run until it fits")
# plt.show()

---
## Part 5 — Build the mesh and run tracking

This places the tracking points, then follows each one across every frame. For a long
video with many points this can take a little while — a progress line shows how far along
it is.

In [ ]:
pts = generate_grid_points(roi_path, mesh_spacing)
tri = triangulate_points(pts)
XA, YA = init_tracks(pts, n_frames, start_frame)
print(f"{len(pts)} tracking points placed inside the ROI.")

XA, YA, CA = track_sequence(
    imgs, pts, XA, YA, start_frame,
    delta_x, delta_y, t_x, t_y, corr_threshold, max_disp,
    tri=tri, fill_gaps=fill_gaps, min_neighbors_fill=min_neighbors,
)
print("\nTracking complete.")

In [ ]:
point_strain = compute_point_strain(pts, tri, XA, YA, start_frame, min_neighbors=min_neighbors)
tri_strain = compute_triangle_strain(tri, point_strain)
print("Strain computed.")

---
## Part 6 — View the results

### Interactive frame slider
Drag the slider to scrub through frames. Colors show the strain (blue = low, red = high).

In [ ]:
import ipywidgets as widgets

data, label = select_strain_component(tri_strain, strain_mode)
vmin = float(np.nanpercentile(data[start_frame:], 5))
vmax = float(np.nanpercentile(data[start_frame:], 95))
cmap = plt.cm.jet


def show_frame(k):
    pts_now = np.array([[YA[i, k], XA[i, k]] for i in range(len(pts))])
    frame_bgr = render_overlay_frame(imgs[k], tri, data[k], pts_now, vmin, vmax, cmap, show_triangle_edges)
    plt.figure(figsize=(9, 9))
    plt.imshow(frame_bgr[:, :, ::-1])  # BGR -> RGB
    plt.title(f"Frame {k}   |   {label}")
    plt.axis("off")
    plt.show()


widgets.interact(
    show_frame,
    k=widgets.IntSlider(min=start_frame, max=n_frames - 1, step=1, value=start_frame, description="Frame"),
);

### Strain over time

In [ ]:
mean_strain = np.nanmean(data, axis=1)
frame_nums = np.arange(n_frames)

plt.figure(figsize=(9, 4))
plt.plot(frame_nums[start_frame:], mean_strain[start_frame:], linewidth=1.5)
plt.xlabel("Frame")
plt.ylabel(f"Mean {label}")
plt.title("Strain Over Time")
plt.grid(alpha=0.3)
plt.show()

---
## Part 7 — Download your results

Writes the CSV (per-point strain + tracking confidence), the strain-over-time plot, and
the overlay video, bundles them into a zip, and downloads it to your computer.

In the CSV, a `confidence` of `-1` means that point's position was **estimated** from its
neighbors (gap-filling) rather than directly measured — filter those out if you only want
directly-measured data.

In [ ]:
import shutil

out_dir = "results"
os.makedirs(out_dir, exist_ok=True)
base_name = os.path.splitext(os.path.basename(VIDEO_FILENAME))[0]

export_strain_csv(out_dir, base_name, point_strain, CA, XA, YA, start_frame)
plot_strain_over_time(out_dir, base_name, point_strain, strain_mode, start_frame)
export_strain_video(out_dir, base_name, imgs, XA, YA, tri, tri_strain, strain_mode, show_triangle_edges, start_frame)

zip_path = shutil.make_archive(base_name + "_results", "zip", out_dir)
print("\nBundled:", zip_path)

from google.colab import files
files.download(zip_path)